# TECTON PNUD — entrenamiento y validación en Colab
Código incluido y verificado por SHA256; el notebook llama al mismo motor Python, no reimplementa el entrenamiento. CPU, Python 3.11–3.13.
No contiene targets ni datos oficiales: los tres CSV se suben en la celda de datos.

Reglas del reto: no compartir los datos fuera del equipo, no editar predicciones a mano, cierre 15:00 hora Colombia por el formulario oficial.

Flujo: ejecutar todo una vez. Para otro experimento, cambiar `CONFIG_NAME` en la primera celda y ejecutar de nuevo desde **Configuración**; los datos ya cargados se reutilizan.


In [ ]:
from pathlib import Path

INSTALL_DEPENDENCIES = True
USE_SYNTHETIC_DATA = False
# None = configuración embebida (la del run de entrega). Otras: 'first_submission', 'baseline', 'catboost'.
CONFIG_NAME = None
# Hilos de CPU; None conserva los de la configuración. Cambiarlos no cambia el protocolo temporal.
THREADS = None
# True guarda runs/ y state/ en el Drive del equipo para sobrevivir desconexiones. No compartir esa carpeta.
USE_DRIVE = False
DRIVE_FOLDER = 'TECTON_PNUD'
# Enlace de Drive a hackathon_datos.zip compartido por el PNUD. Pegarlo solo en la copia de Colab del equipo; no versionarlo.
URL_DATOS = ''
# Alternativa: subir hackathon_datos.zip con el ícono de carpeta, o los tres CSV cuando se soliciten.
DATA_ZIP = Path('hackathon_datos.zip')
LOCAL_DATA_DIR = Path('data/raw')


In [ ]:
try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if USE_DRIVE and IN_COLAB:
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive') / DRIVE_FOLDER / 'tecton_colab'
else:
    ROOT = Path.cwd() / 'tecton_colab'
ROOT.mkdir(parents=True, exist_ok=True)
print('Proyecto:', ROOT)


In [ ]:
import base64, hashlib, io, json, shutil, zipfile
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == 'd23acaf0aa8453e54814052ed007068c8e76f3626d2168006837b6e59c854f26'
# Reemplazar el código de una versión anterior; runs/, state/ y data/ se conservan.
for folder in ['src', 'configs', 'docs']:
    shutil.rmtree(ROOT / folder, ignore_errors=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for item in z.infolist():
        p = Path(item.filename)
        assert not p.is_absolute() and '..' not in p.parts
    z.extractall(ROOT)


In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

if not ((3, 11) <= sys.version_info[:2] < (3, 14)):
    raise RuntimeError('Este scaffold requiere Python 3.11–3.13; seleccionar un runtime compatible.')
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements-colab.txt')])
pins = {}
for line in (ROOT / 'requirements-colab.txt').read_text().splitlines():
    if '==' in line and not line.startswith('#'):
        name, version = line.split(';')[0].strip().split('==')
        pins[name.lower()] = version
stale = []
for name, module in [('numpy', 'numpy'), ('pandas', 'pandas'), ('scikit-learn', 'sklearn'), ('catboost', 'catboost')]:
    loaded = sys.modules.get(module)
    found = getattr(loaded, '__version__', None) if loaded else metadata.version(name)
    if found != pins[name]:
        stale.append(f'{name} {found} != {pins[name]}')
if stale:
    raise RuntimeError('Versiones distintas del lock: ' + ', '.join(stale) + '. Reiniciar la sesión y ejecutar desde el inicio.')
# Descargar módulos tecton de una versión anterior del código en esta misma sesión.
for module in [m for m in sys.modules if m == 'tecton' or m.startswith('tecton.')]:
    del sys.modules[module]
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('Dependencias del lock:', {n: pins[n] for n in ['numpy', 'pandas', 'scikit-learn', 'catboost']})


## Datos
Igual que el notebook base del PNUD: pegar `URL_DATOS` o subir `hackathon_datos.zip`. Si no hay ZIP, se piden los tres CSV con su nombre exacto. Se cargan una vez por sesión (o una sola vez si `USE_DRIVE=True`). No se imprimen filas. Para un ensayo, activar `USE_SYNTHETIC_DATA` en la primera celda.


In [ ]:
from tecton.demo import generate

NAMES = ['entrenamiento.csv', 'prueba_equipos.csv', 'prueba_oculta.csv']
if USE_SYNTHETIC_DATA:
    generate(ROOT / 'data/synthetic')
    print('Datos INVENTADOS en data/synthetic; solo ensayo.')
else:
    raw = ROOT / 'data/raw'
    raw.mkdir(parents=True, exist_ok=True)
    missing = [name for name in NAMES if not (raw / name).exists()]
    if missing and URL_DATOS and not DATA_ZIP.exists():
        import re
        import gdown
        # Usar el id del archivo: compatible con versiones de gdown con y sin el argumento fuzzy.
        file_id = re.search(r'/d/([A-Za-z0-9_-]+)', URL_DATOS) or re.search(r'id=([A-Za-z0-9_-]+)', URL_DATOS)
        gdown.download('https://drive.google.com/uc?id=' + (file_id.group(1) if file_id else URL_DATOS), str(DATA_ZIP), quiet=True)
    if missing and DATA_ZIP.exists():
        # El ZIP oficial puede traer subcarpetas; se copian los CSV por nombre exacto, sin modificarlos.
        with zipfile.ZipFile(DATA_ZIP) as z:
            for item in z.infolist():
                if Path(item.filename).name in NAMES + ['diccionario_datos.csv'] and not item.is_dir():
                    (raw / Path(item.filename).name).write_bytes(z.read(item))
        missing = [name for name in NAMES if not (raw / name).exists()]
    if missing and IN_COLAB:
        uploaded = files.upload()
        for name in missing:
            if name not in uploaded:
                raise ValueError('Falta el archivo con nombre exacto: ' + name)
            (raw / name).write_bytes(uploaded[name])
    else:
        for name in missing:
            shutil.copy2(LOCAL_DATA_DIR / name, raw / name)
    print('CSV oficiales listos en', raw)


## Configuración y auditoría
Solo resúmenes agregados: filas, municipios, fechas, nulos y prevalencia.


In [ ]:
from tecton.pipeline import dump_json
from tecton.schema import load_inputs

EMBEDDED_CONFIG = json.loads('{"name": "first-provisional", "data_dir": "data/raw", "strict": true, "seed": 42, "threads": 2, "model": {"family": "hist", "iterations": 60, "learning_rate": 0.08, "depth": 5}, "features": {"use_history": true, "smoothing": 20.0}, "stress_test": false, "folds": [{"train_end": "2021-09-01", "valid_start": "2021-10-01", "valid_end": "2022-09-01"}]}')
SMOKE = {None: 'smoke', 'first_submission': 'smoke', 'baseline': 'baseline-smoke', 'catboost': 'catboost-smoke'}
if USE_SYNTHETIC_DATA:
    config = json.loads((ROOT / 'configs' / f"{SMOKE.get(CONFIG_NAME, CONFIG_NAME)}.json").read_text())
elif CONFIG_NAME:
    config = json.loads((ROOT / 'configs' / f'{CONFIG_NAME}.json').read_text())
else:
    config = EMBEDDED_CONFIG
if config['data_dir'] != ('data/synthetic' if USE_SYNTHETIC_DATA else 'data/raw'):
    raise ValueError('La configuración no corresponde al modo de datos elegido.')
if THREADS:
    config = {**config, 'threads': int(THREADS)}
_, audit, _, synthetic = load_inputs(ROOT / config['data_dir'], config['strict'])
dump_json(ROOT / 'configs/colab_run.json', config)
print(json.dumps({'config': config['name'], 'threads': config['threads'], 'synthetic': synthetic, 'audit': audit}, indent=2, ensure_ascii=False))


## Entrenamiento
Imprime AUC, RMSE-log, cobertura y segundos por fold. Cada run queda en una carpeta nueva; un fallo no reemplaza el champion.


In [ ]:
from tecton.pipeline import run

run_id = run(ROOT, ROOT / 'configs/colab_run.json')
print('Experimento:', run_id)


In [ ]:
import pandas as pd
from IPython.display import HTML, display
from tecton.pipeline import compare_runs

pointer = ROOT / 'state' / ('champion-demo.json' if USE_SYNTHETIC_DATA else 'champion.json')
print('Champion:', json.loads(pointer.read_text())['run_id'] if pointer.exists() else 'sin champion')
display(pd.DataFrame(compare_runs(ROOT)))


## Promoción
Escribir el `run_id` y ejecutar. El harness rechaza runs incompletos, sin cinco folds/stress o que no superan al champion bajo el mismo protocolo.


In [ ]:
PROMOTE_RUN_ID = None
PROMOTE_REASON = 'Cinco folds, stress de 12 meses y QA revisados'
if PROMOTE_RUN_ID:
    from tecton.pipeline import promote
    print(json.dumps(promote(ROOT, PROMOTE_RUN_ID, PROMOTE_REASON, demo=USE_SYNTHETIC_DATA), indent=2, ensure_ascii=False))


In [ ]:
display(HTML((ROOT / 'runs' / run_id / 'dashboard.html').read_text()))


## Descargas
1. **Resumen para revisión**: métricas, manifiestos, auditoría y comparación. Sin filas, sin OOF ni predicciones; es lo que se comparte con el asistente de código.
2. **Entrega**: CSV, notebook reproducible, replicabilidad y declaraciones. Va al formulario oficial.
3. **Run para el geovisor**: el run sin `oof.csv`, para importarlo en la PC con `tecton import-run`.


In [ ]:
from tecton.artifacts import review_pack

review = review_pack(ROOT)
print('Resumen para revisión:', review)
if IN_COLAB:
    files.download(str(review))


In [ ]:
EXPORT_RUN_ID = None  # None = champion si existe; si no, el último run de esta sesión.
from tecton.artifacts import bundle, run_pack, zip_folder

selected = EXPORT_RUN_ID or (json.loads(pointer.read_text())['run_id'] if pointer.exists() else globals().get('run_id'))
if not selected:
    raise ValueError('Indicar EXPORT_RUN_ID.')
delivery = ROOT / 'delivery' / selected
if not delivery.exists():
    bundle(ROOT, run_id=selected, demo=USE_SYNTHETIC_DATA)
delivery_zip = zip_folder(delivery, ROOT / 'export' / f'entrega-{selected}.zip')
run_zip = run_pack(ROOT, selected)
print('Entrega:', delivery_zip)
print('Run para geovisor:', run_zip)
print('Completar ai_usage.csv y revisar replicabilidad antes de enviar. El formulario lo envía el equipo.')
if IN_COLAB:
    files.download(str(delivery_zip))
    files.download(str(run_zip))
